In [1]:
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw')
print(pd.__version__)

3.0.6


## Preparación: carga de datos

In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

In [3]:
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


# Lección 2: Limpieza y estandarización de datos crudos

En esta lección se resuelven los problemas encontrados en la exploración: duplicados, tipos de dato incorrectos, nulos, texto sin estandarizar y duplicados implícitos.

Para cada problema hay tres opciones: **eliminar**, **corregir** o **conservar**, y la decisión depende de lo que significa el dato.

Orden de trabajo: duplicados → tipos → nulos → texto → duplicados implícitos → verificación.

## 1. Duplicados exactos
Primero se **detectan** con `duplicated()` (solo marca, no borra) y luego se **eliminan** con `drop_duplicates()`.

In [4]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


In [5]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


**Resultado:** 30 filas marcadas = 15 pedidos duplicados. Todos ocurren el miércoles (`order_dow = 3`) a las 2 a. m.
**Decisión:** eliminar.
**Justificación:** que todos caigan en la misma franja indica un fallo del sistema, no pedidos reales.

## 2. Tipos de dato con `astype`
Se cambian los enteros de 64 bits por tipos más pequeños según el rango de cada columna, para reducir la memoria.

In [6]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


**Resultado:** la memoria de `orders` bajó de 23,0 MB a 7,2 MB.
Las columnas con nulos (`days_since_prior_order` y `add_to_cart_order`) usan `Int16` (mayúscula) porque es el tipo entero que admite valores nulos.

## 3. Nulos: comprobar la causa antes de decidir
Un nulo puede significar "no aplica", "no se registró" o "error".

In [7]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


**Decisión:** conservar los nulos de `days_since_prior_order`.
**Justificación:** cada nulo es un primer pedido; no existe pedido anterior. Rellenar con 0 diría que el cliente volvió el mismo día y bajaría la media de espera.

In [8]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


**Decisión:** rellenar `product_name` con `'Unknown'`. Los 1.258 productos están en el pasillo 100 y el departamento 21 (`missing`); el producto existe y aparece en pedidos, no se puede borrar.

| Decisión | Columna | Causa |
|---|---|---|
| Conservar el nulo | `days_since_prior_order` | Primer pedido del cliente |
| Rellenar con `'Unknown'` | `product_name` | Producto sin datos de catálogo |
| Conservar el nulo | `add_to_cart_order` | Se investiga en el reto |

## 4. Estandarización de texto con `.str`

In [9]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


**Resultado:** 403 nombres tenían espacios dobles y 16 un espacio invisible (`\xa0`). Tras la limpieza quedan 0 y 0.
Se reemplazó el espacio invisible, se unificaron los espacios, se quitaron los de los extremos y se puso mayúscula inicial en cada palabra.

## 5. Duplicados implícitos
Nombres que solo se repiten después de estandarizar el texto.

In [10]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


**Decisión:** conservar y documentar.
**Justificación:** tienen `product_id` distintos y 268 de 280 aparecen en pedidos; borrarlos dejaría pedidos apuntando a productos inexistentes. En los análisis se agrupará por `product_id`, no por nombre.

## 6. Limpieza reutilizable con funciones

In [11]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


In [12]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

## Resumen
1. Se revisaron y eliminaron 15 pedidos duplicados con `duplicated(keep=False)` y `drop_duplicates`.
2. Se corrigieron los tipos con `astype`, usando `Int16` en las columnas con nulos.
3. Se analizó la causa de los nulos: se conservaron en `days_since_prior_order` y se rellenó `product_name` con `'Unknown'`.
4. Se estandarizó el texto con `.str.replace`, `.str.strip` y `.str.title`.
5. Se detectaron 141 duplicados implícitos y se conservaron porque se usan en pedidos.
6. Se empaquetó la limpieza en funciones y se verificó con `assert`.

# Laboratorio guiado 2: Pipeline de limpieza con informe de cambios

Se construye un pipeline: una secuencia de pasos de limpieza encadenados que parte del archivo crudo, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso.

## Paso 1: Recargar los datos crudos

In [13]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


## Paso 2: Función que anota el estado de la tabla
Después de cada paso se toma una "foto" de la tabla: filas, nulos y memoria.

In [14]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

## Paso 3: Un paso de limpieza por función
Cada función hace una sola cosa: recibe un DataFrame y devuelve otro.

In [15]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

## Paso 4: Encadenar los pasos con `pipe`

In [16]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

## Paso 5: Informe de cambios

In [17]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,5.1
4,products,1 nombres,49694,0,5.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**
- **Quitar duplicados** eliminó 15 filas de `orders` y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- **Corregir tipos** no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- **Limpiar nombres** dejó `products` con 0 nulos (los 1.258 se rellenaron con `'Unknown'`).
- Los **836 nulos** de `order_prods` siguen ahí a propósito: se decidió conservarlos.

## Paso 6: Validar el resultado
El pipeline termina con una comprobación automática de las reglas que deben cumplir los datos limpios.

In [18]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


## Conclusiones clave
- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- Intercalar una función de registro (`anotar`) produce un informe de cambios sin esfuerzo extra.
- El informe responde con cifras a la pregunta "¿qué se le hizo a los datos?".
- Validar con `assert` al final convierte las reglas de calidad en una prueba que se ejecuta cada vez.

# Reto 2

## Ejercicio 1
Comprueba si `order_prods` tiene filas duplicadas exactas, si se repite algún par `order_id` + `product_id` y si `reordered` solo contiene 0 y 1.

In [19]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', order_prods['reordered'].unique())

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0 1]


**Resultado:** 0 filas duplicadas exactas y 0 pares `order_id` + `product_id` repetidos; `reordered` solo contiene 0 y 1.
**Conclusión:** `order_prods` no necesita limpieza de duplicados: cada producto aparece una sola vez por pedido, y `reordered` es una variable binaria válida.

## Ejercicio 2
Comprueba si `aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?

In [20]:
print('Pasillos duplicados:', aisles['aisle'].duplicated().sum())
print('Pasillos con espacios en los extremos:', (aisles['aisle'] != aisles['aisle'].str.strip()).sum())

print('Departamentos duplicados:', departments['department'].duplicated().sum())
print('Departamentos con espacios en los extremos:', (departments['department'] != departments['department'].str.strip()).sum())

Pasillos duplicados: 0
Pasillos con espacios en los extremos: 0
Departamentos duplicados: 0
Departamentos con espacios en los extremos: 0


**Resultado:** `aisles` tiene 0 nombres duplicados y 0 con espacios en los extremos; `departments` también 0 y 0.
**Conclusión:** no necesitan limpieza de texto. Son tablas de referencia pequeñas (134 y 21 filas) que ya vienen limpias.

## Ejercicio 3
Investiga los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [21]:
filas_nulas = order_prods[order_prods['add_to_cart_order'].isna()]
print('Filas con nulo:', len(filas_nulas))

pedidos_con_nulo = filas_nulas['order_id'].unique()
print('Pedidos con nulo:', len(pedidos_con_nulo))

detalle = order_prods[order_prods['order_id'].isin(pedidos_con_nulo)]
productos_por_pedido = detalle.groupby('order_id').size()
print('Mínimo de productos en esos pedidos:', productos_por_pedido.min())
print('Posición máxima registrada en todo order_prods:', order_prods['add_to_cart_order'].max())

Filas con nulo: 836
Pedidos con nulo: 70
Mínimo de productos en esos pedidos: 65
Posición máxima registrada en todo order_prods: 64


**Resultado:** los 836 nulos ocurren en 70 pedidos, y cada uno tiene como mínimo 65 productos.
**Explicación:** la posición más alta registrada en toda la tabla es 64. Todos los pedidos con nulos superan los 64 productos, así que el sistema solo guardaba la posición en el carrito hasta el producto 64; desde el 65 en adelante quedó vacía.
**Decisión:** conservar el nulo. El producto sí fue comprado; lo que falta es solo su posición, y no hay forma confiable de reconstruirla.

## Ejercicio 4
Carga de nuevo `products.csv` en una variable `products_crudo` y cuenta cuántos nombres (no nulos) cambiaron con la estandarización.

In [22]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

no_nulos = products_crudo['product_name'].notna()
cambiaron = (products_crudo.loc[no_nulos, 'product_name'] != products.loc[no_nulos, 'product_name']).sum()
print('Nombres que cambiaron:', cambiaron)

Nombres que cambiaron: 6321


**Resultado:** 6.321 nombres cambiaron con la estandarización (espacios dobles, espacios invisibles, espacios en los extremos o mayúsculas).
Se excluyeron los nulos porque los 1.258 productos sin nombre ahora dicen `'Unknown'` y se contarían como cambio sin que hubiera un nombre que corregir.

## Ejercicio 5
Crea una Series con el nombre del día de cada pedido y conviértela a una categoría ordenada de domingo a sábado. Compara la memoria del texto con la de la categoría.|

In [23]:
dias = {0: 'Domingo', 1: 'Lunes', 2: 'Martes', 3: 'Miércoles',
        4: 'Jueves', 5: 'Viernes', 6: 'Sábado'}

dia_texto = orders['order_dow'].map(dias)

orden = ['Domingo', 'Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado']
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=orden, ordered=True))

print('Texto:', round(dia_texto.memory_usage(deep=True) / 1e6, 1), 'MB')
print('Categoría:', round(dia_categoria.memory_usage(deep=True) / 1e6, 1), 'MB')
dia_categoria.head()

Texto: 28.6 MB
Categoría: 0.5 MB


0     Sábado
1    Viernes
2    Viernes
3    Domingo
4     Jueves
dtype: category
Categories (7, str): ['Domingo' < 'Lunes' < 'Martes' < 'Miércoles' < 'Jueves' < 'Viernes' < 'Sábado']

**Resultado:** como texto ocupa 28.6 MB; como categoría ocupa 0,5 MB, unas 57.2 veces menos.
**Explicación:** el texto guarda la palabra completa en cada una de las 478.952 filas. La categoría guarda los 7 nombres una sola vez y en cada fila solo un número pequeño que apunta a uno de ellos. Además, al ser ordenada, al agrupar u ordenar los días aparecen de domingo a sábado y no en orden alfabético.

## Ejercicio 6
¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica.

In [24]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']

filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Filas:', len(filas_unknown))
print('Pedidos:', filas_unknown['order_id'].nunique())

Filas: 9547
Pedidos: 8254


**Resultado:** 9.547 filas de `order_prods` corresponden a productos `'Unknown'`, repartidas en 8.254 pedidos.
**¿Los eliminaría?:** no.
**Justificación:** son compras reales; al producto solo le falta el nombre en el catálogo. Eliminarlos dejaría 8.254 pedidos incompletos y reduciría el total de artículos, la tasa de recompra y el tamaño de los carritos. Es mejor conservarlos con la etiqueta `'Unknown'` y excluirlos solo en los análisis que dependan del nombre del producto.

## Ejercicio 7
En una copia de `departments`, reemplaza `'missing'` por un nulo real (`pd.NA`). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.

In [25]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)

print(departments_copia.tail(3))
print('Original con missing:', (departments['department'] == 'missing').sum())

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Original con missing: 1


**Explicación:** no conviene hacerlo en la tabla original porque `groupby` descarta los nulos por defecto (`dropna=True`): los 1.258 productos del departamento 21 desaparecerían en silencio de cualquier resumen por departamento, mientras que con la etiqueta `'missing'` siguen apareciendo como un grupo visible.